In [1]:
# Cellule 1 — imports

import json
import requests
from pprint import pprint

In [14]:
# Cellule 2 — configuration Ollama

OLLAMA_BASE_URL = "http://127.0.0.1:11434"
MODEL_NAME = "qwen3:1.7b"   # adaptez si votre tag local diffère
TIMEOUT_S = 120

In [23]:
# Cellule 3 — fonction d'appel simple à Ollama (/api/generate)

def ollama_generate(
    prompt: str,
    model: str = MODEL_NAME,
    system: str | None = None,
    temperature: float = 0.0,
    num_predict: int = 512,
    base_url: str = OLLAMA_BASE_URL,
    timeout_s: int = TIMEOUT_S,
):
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0,
            "top_p": 1,
            "top_k": 1,
            "repeat_penalty": 1.0,
            "seed": 42,
            "think": False
        }
    }
    if system is not None:
        payload["system"] = system

    r = requests.post(
        f"{base_url}/api/generate",
        json=payload,
        timeout=timeout_s,
    )
    r.raise_for_status()
    data = r.json()
    return data["response"]

In [16]:
# Cellule 4 — prompt système pour extraction structurée

SYSTEM_PROMPT = """Vous êtes un extracteur de structure documentaire.

Votre tâche est de transformer un texte brut en un objet JSON valide.

Vous devez répondre UNIQUEMENT par un objet JSON.
Vous ne devez produire aucun commentaire, aucune explication, aucun markdown.

Schéma de sortie :
{
  "type": "note_rapide" | "texte_libre" | "note_structuree",
  "titre": string | null,
  "titre_source": "explicite" | "incipit" | null,
  "mots_clefs": [string, ...],
  "date": string | null,
  "texte": string
}

Règles :
- Retournez toujours un JSON valide.
- N'inventez jamais de champ hors schéma.
- Si un champ est absent, utilisez null pour les scalaires et [] pour mots_clefs.
- Le champ "texte" doit contenir le corps principal sans les métadonnées de tête.
- Si un titre explicite est présent, utilisez-le et mettez "titre_source": "explicite".
- Si aucun titre explicite n'est présent mais qu'un titre court peut être déduit par incipit, produisez-le et mettez "titre_source": "incipit".
- N'inventez pas de mots-clefs absents.
- Si des hashtags ou mots-clefs explicites sont présents, recopiez-les.
- Ne résumez pas le texte.
- Ne reformulez pas le texte.
- Conservez les accents et la ponctuation autant que possible.
- Le type "note_rapide" convient à un texte court ou moyen avec éventuellement quelques métadonnées simples.
- Le type "note_structuree" convient si plusieurs champs explicites sont présents.
- Le type "texte_libre" convient si aucune structure documentaire claire n'est détectable.

Exemples :

Entrée :
Le Dormeur du val est un sonnet en alexandrins d'Arthur Rimbaud.

Sortie :
{"type":"note_rapide","titre":"Le Dormeur du val","titre_source":"incipit","mots_clefs":[],"date":null,"texte":"Le Dormeur du val est un sonnet en alexandrins d'Arthur Rimbaud."}

Entrée :
#poésie, #Rimbaud, #guerre C’est un trou de verdure où chante une rivière.

Sortie :
{"type":"note_rapide","titre":"C’est un trou de verdure","titre_source":"incipit","mots_clefs":["#poésie","#Rimbaud","#guerre"],"date":null,"texte":"C’est un trou de verdure où chante une rivière."}
"""

In [17]:
# Cellule 5 — construction du prompt utilisateur

def build_user_prompt(text: str) -> str:
    return f"""Analysez le texte suivant et extrayez sa structure documentaire.

Texte :
{text}
"""

In [18]:
# Cellule 6 — helper de parsing JSON robuste

def extract_json_object(raw: str):
    raw = raw.strip()

    # cas nominal
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        pass

    # tentative de récupération du premier bloc {...}
    start = raw.find("{")
    end = raw.rfind("}")
    if start != -1 and end != -1 and end > start:
        candidate = raw[start:end+1]
        return json.loads(candidate)

    raise ValueError(f"Impossible de parser un JSON valide depuis :\n{raw}")

In [19]:
# Cellule 7 — validation minimale du schéma

ALLOWED_TYPES = {"note_rapide", "texte_libre", "note_structuree"}
ALLOWED_TITRE_SOURCE = {"explicite", "incipit", None}

def validate_structure(obj: dict):
    required = {"type", "titre", "titre_source", "mots_clefs", "date", "texte"}
    missing = required - set(obj.keys())
    extra = set(obj.keys()) - required

    if missing:
        raise ValueError(f"Champs manquants : {missing}")
    if extra:
        raise ValueError(f"Champs inattendus : {extra}")

    if obj["type"] not in ALLOWED_TYPES:
        raise ValueError(f"type invalide : {obj['type']}")

    if obj["titre_source"] not in ALLOWED_TITRE_SOURCE:
        raise ValueError(f"titre_source invalide : {obj['titre_source']}")

    if not isinstance(obj["mots_clefs"], list):
        raise ValueError("mots_clefs doit être une liste")

    if not isinstance(obj["texte"], str) or not obj["texte"].strip():
        raise ValueError("texte doit être une chaîne non vide")

    return True

In [20]:
# Cellule 8 — fonction principale d'extraction

def extract_document_structure(
    text: str,
    model: str = MODEL_NAME,
    temperature: float = 0.0,
):
    prompt = build_user_prompt(text)
    raw = ollama_generate(
        prompt=prompt,
        model=model,
        system=SYSTEM_PROMPT,
        temperature=temperature,
        num_predict=512,
    )
    obj = extract_json_object(raw)
    validate_structure(obj)
    return obj, raw

In [21]:
# Cellule 9 — test 1

texte1 = """Le Dormeur du val est un sonnet en alexandrins d'Arthur Rimbaud. Ce poème est le deuxième poème du second Cahier de Douai (ou Recueil Demeny). Le manuscrit autographe, daté d'octobre 1870, est conservé à la British Library[1]. Il n'existe pas d'autre manuscrit connu."""

In [24]:
# Cellule 10 — exécution test 1

obj1, raw1 = extract_document_structure(texte1)
print(raw1)
pprint(obj1)

{
  "type": "note_structuree",
  "titre": "Le Dormeur du val",
  "titre_source": "incipit",
  "mots_clefs": ["sonnet", "alexandrins", "Arthur Rimbaud", "Cahier de Douai", "manuscrit autographe", "British Library", "octobre 1870"],
  "date": "octobre 1870",
  "texte": "Le Dormeur du val est un sonnet en alexandrins d'Arthur Rimbaud. Ce poème est le deuxième poème du second Cahier de Douai (ou Recueil Demeny). Le manuscrit autographe, daté d'octobre 1870, est conservé à la British Library[1]. Il n'existe pas d'autre manuscrit connu."
}
{'date': 'octobre 1870',
 'mots_clefs': ['sonnet',
                'alexandrins',
                'Arthur Rimbaud',
                'Cahier de Douai',
                'manuscrit autographe',
                'British Library',
                'octobre 1870'],
 'texte': "Le Dormeur du val est un sonnet en alexandrins d'Arthur Rimbaud. Ce "
          'poème est le deuxième poème du second Cahier de Douai (ou Recueil '
          "Demeny). Le manuscrit autograp

In [ ]:
# Cellule 11 — test 2

texte2 = """#poésie, #Rimbaud, #guerre C’est un trou de verdure où chante une rivière Accrochant follement aux herbes des haillons D’argent ; où le soleil, de la montagne fière, Luit : c’est un petit val qui mousse de rayons."""

In [ ]:
# Cellule 12 — exécution test 2

obj2, raw2 = extract_document_structure(texte2)
print(raw2)
pprint(obj2)

In [ ]:
# Cellule 13 — batterie de tests

tests = [
    "Titre : Le dormeur du val\nMots clefs : #poésie, #Rimbaud, #guerre\nC’est un trou de verdure où chante une rivière.",
    "titre le dormeur du val\nmots-clés #poésie #Rimbaud\nC’est un trou de verdure où chante une rivière.",
    "Tags : #poésie, #Rimbaud\nLe Dormeur du val est un sonnet en alexandrins d'Arthur Rimbaud.",
    "Le Dormeur du val est un sonnet en alexandrins d'Arthur Rimbaud.",
]

results = []
for i, t in enumerate(tests, 1):
    try:
        obj, raw = extract_document_structure(t)
        results.append({
            "id": i,
            "ok": True,
            "raw": raw,
            "parsed": obj,
        })
    except Exception as e:
        results.append({
            "id": i,
            "ok": False,
            "error": str(e),
        })

results